# 🔍 UX Research Intelligence Tool

Turn raw interview transcripts into structured **insights, themes, quotes, and recommendations**.

**Powered by:** Groq (free LLM API) + local sentence embeddings + Qdrant vector search.

---

## How to use this notebook

1. **Cell 1** — Install dependencies (~60 sec, one-time)
2. **Cell 2** — Paste your free Groq API key
3. **Cell 3** — Load all core code (just run it)
4. **Cell 4** — Load the sample study or paste your own transcripts
5. **Cell 5** — Run the full 6-stage analysis pipeline
6. **Cell 6** — View your results
7. **Cell 7** — Ask questions about your transcripts (semantic search)
8. **Cell 8** — Submit feedback to improve future analyses
9. **Cell 9** — Export report as JSON and download it
10. **Cell 10** — Add more participants and re-run

> Get your **free** Groq API key at https://console.groq.com (no credit card needed)

---
## Cell 1 — Install Dependencies

In [1]:
print("Installing packages... (~60 seconds on first run)")
!pip install -q groq sentence-transformers "qdrant-client==1.6.9" scikit-learn numpy
print("Done! All packages installed.")

Installing packages... (~60 seconds on first run)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.2/182.2 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.5/323.5 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 15.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.5 whic

---
## Cell 2 — Set Your Groq API Key

Get your free key at: https://console.groq.com

In [2]:
import os
from getpass import getpass

print("Get your FREE Groq API key at: https://console.groq.com")
print("Free tier, no credit card needed.\n")

GROQ_API_KEY = getpass("Enter you Grok API key here")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

GROQ_MODEL = "llama-3.3-70b-versatile"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

print("API key saved! Model:", GROQ_MODEL)

Get your FREE Groq API key at: https://console.groq.com
Free tier, no credit card needed.

Enter you Grok API key here··········
API key saved! Model: llama-3.3-70b-versatile


---
## Cell 3 — Load All Core Code

Just run this cell. It loads all models, pipeline stages, and helpers.

In [6]:
# ================================================================
# PART A — DATA MODELS
# ================================================================
from dataclasses import dataclass, field
from typing import Optional
from enum import Enum
import uuid, json, re, os
from datetime import datetime, timezone

def uid():
    return str(uuid.uuid4())[:8]

@dataclass
class Participant:
    pseudonym: str
    tags: dict = field(default_factory=dict)
    id: str = field(default_factory=uid)

@dataclass
class Utterance:
    speaker: str
    text: str
    timestamp: Optional[str] = None

@dataclass
class Transcript:
    participant: Participant
    raw_text: str
    utterances: list = field(default_factory=list)

@dataclass
class Study:
    study_purpose: str
    transcripts: list
    research_questions: list = field(default_factory=list)
    script_questions: list = field(default_factory=list)
    study_id: str = field(default_factory=lambda: str(uuid.uuid4())[:12])

@dataclass
class Insight:
    participant_id: str
    pseudonym: str
    participant_tags: dict
    insight_type: str
    description: str
    quote: Optional[str] = None
    question_context: Optional[str] = None
    is_surprising: bool = False
    id: str = field(default_factory=uid)

@dataclass
class Theme:
    name: str
    description: str
    participant_count: int
    confidence: str
    insight_types: list
    evidence: list
    representative_quote: Optional[str] = None
    representative_quote_participant: Optional[str] = None
    id: str = field(default_factory=uid)

@dataclass
class Recommendation:
    title: str
    description: str
    rec_type: str
    priority: str
    evidence_strength: str
    linked_theme_names: list = field(default_factory=list)
    id: str = field(default_factory=uid)

@dataclass
class ResearchGap:
    description: str
    gap_type: str
    related_question: Optional[str] = None
    suggested_follow_up: Optional[str] = None

@dataclass
class Contradiction:
    topic: str
    description: str
    participant_a: str
    participant_b: str
    quote_a: Optional[str] = None
    quote_b: Optional[str] = None

@dataclass
class StudyReport:
    study_id: str
    study_purpose: str
    participant_count: int
    executive_summary: str
    themes: list
    recommendations: list
    gaps: list
    contradictions: list
    all_insights: list
    generated_at: str = field(default_factory=lambda: datetime.now(tz=timezone.utc).isoformat())

print("Data models loaded.")

# ================================================================
# PART B — TRANSCRIPT PARSER
# ================================================================

def parse_transcript(raw_text, pseudonym):
    utterances = []
    ts_pattern = re.compile(r"\[(\d{1,2}:\d{2}(?::\d{2})?)\]\s+([^:]+?):\s+(.+)", re.IGNORECASE)
    lb_pattern = re.compile(r"^([A-Z][A-Za-z\s]{1,30}):\s+(.+)", re.MULTILINE)
    MOD = {"moderator","interviewer","researcher","facilitator","mod","int","r","q"}

    def norm(s):
        s = s.strip().lower()
        return "moderator" if s in MOD or s.startswith("mod") or s.startswith("int") else "participant"

    for line in [l.strip() for l in raw_text.strip().splitlines() if l.strip()]:
        ts = ts_pattern.match(line)
        lb = lb_pattern.match(line)
        if ts:
            t, spk, txt = ts.groups()
            utterances.append(Utterance(speaker=norm(spk), text=txt.strip(), timestamp=t))
        elif lb:
            spk, txt = lb.groups()
            utterances.append(Utterance(speaker=norm(spk), text=txt.strip()))
        else:
            utterances.append(Utterance(speaker="participant", text=line))
    return utterances

def transcript_to_text(transcript):
    lines = []
    for u in transcript.utterances:
        label = "Moderator" if u.speaker == "moderator" else transcript.participant.pseudonym
        ts = " [" + u.timestamp + "]" if u.timestamp else ""
        lines.append(label + ts + ": " + u.text)
    return "\n".join(lines)

print("Transcript parser loaded.")

# ================================================================
# PART C — LLM CLIENT (GROQ)
# ================================================================
from groq import Groq

groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])

def llm_complete(system, user, temperature=0.2, max_tokens=4096):
    resp = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role":"system","content":system},{"role":"user","content":user}],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return resp.choices[0].message.content.strip()

def llm_json(system, user, temperature=0.1, max_tokens=4096, retries=2):
    def strip_fences(t):
        t = t.strip()
        t = re.sub(r"^```(?:json)?\s*\n?", "", t)
        t = re.sub(r"\n?```\s*$", "", t)
        return t.strip()
    _user = user
    for attempt in range(retries + 1):
        try:
            raw = llm_complete(system, _user, temperature, max_tokens)
            return json.loads(strip_fences(raw))
        except json.JSONDecodeError:
            _user = user + "\n\nReturn ONLY valid JSON. No markdown, no explanation."
    raise ValueError("LLM returned invalid JSON after retries")

print("LLM client loaded.")

# ================================================================

import json as _json

def safe_parse_insights(raw_text):
    """Try to parse insights JSON. Accepts {"insights":[...]} or a bare list."""
    try:
        data = _json.loads(raw_text)
    except _json.JSONDecodeError:
        return []
    if isinstance(data, dict) and "insights" in data and isinstance(data["insights"], list):
        return data["insights"]
    if isinstance(data, list):
        return data
    return []

def safe_parse_themes(raw_text):
    """Try to parse themes JSON."""
    try:
        data = _json.loads(raw_text)
    except _json.JSONDecodeError:
        return []
    if isinstance(data, dict) and "themes" in data and isinstance(data["themes"], list):
        return data["themes"]
    if isinstance(data, list):
        return data
    return []

def safe_parse_recommendations(raw_text):
    """Try to parse recommendations JSON."""
    try:
        data = _json.loads(raw_text)
    except _json.JSONDecodeError:
        return []
    if isinstance(data, dict) and "recommendations" in data and isinstance(data["recommendations"], list):
        return data["recommendations"]
    if isinstance(data, list):
        return data
    return []

def safe_parse_gaps(raw_text):
    """Try to parse gaps/contradictions JSON."""
    try:
        data = _json.loads(raw_text)
    except _json.JSONDecodeError:
        return {}, {}
    gaps = data.get("gaps", []) if isinstance(data, dict) else []
    contradictions = data.get("contradictions", []) if isinstance(data, dict) else []
    return gaps, contradictions

# PART D — EMBEDDINGS + VECTOR SEARCH
# ================================================================
print("Loading embedding model (first time ~30 sec)...")
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    Filter, FieldCondition, MatchValue
)

_encoder = SentenceTransformer(EMBEDDING_MODEL)
_qdrant  = QdrantClient(":memory:")
_COLL    = "ux_chunks"

_qdrant.create_collection(
    collection_name=_COLL,
    vectors_config=VectorParams(size=384, distance=Distance.COSINE)
)

def embed_texts(texts):
    return _encoder.encode(texts, show_progress_bar=False).tolist()

def store_chunks(study_id, transcripts):
    points = []
    for t in transcripts:
        for i, u in enumerate(t.utterances):
            if u.speaker != "participant":
                continue
            start = max(0, i - 2)
            end   = min(len(t.utterances), i + 3)
            window = t.utterances[start:end]
            chunk_text = "\n".join(
                ("Moderator" if w.speaker=="moderator" else t.participant.pseudonym) + ": " + w.text
                for w in window
            )
            q_ctx = None
            for j in range(i-1, max(-1, i-5), -1):
                if t.utterances[j].speaker == "moderator":
                    q_ctx = t.utterances[j].text
                    break
            cid = str(uuid.uuid5(uuid.NAMESPACE_DNS, str(uuid.uuid4())))
            points.append(PointStruct(
                id=cid,
                vector=embed_texts([chunk_text])[0],
                payload={
                    "study_id":       study_id,
                    "participant_id": t.participant.id,
                    "pseudonym":      t.participant.pseudonym,
                    "tags":           t.participant.tags,
                    "text":           chunk_text,
                    "question_ctx":   q_ctx,
                }
            ))
    _qdrant.upsert(collection_name=_COLL, points=points)
    print("  Stored", len(points), "chunks in vector DB")


def semantic_search(query, study_id, top_k=8):
    vec = embed_texts([query])[0]
    from qdrant_client.models import Filter, FieldCondition, MatchValue
    hits = _qdrant.query_points(
        collection_name=_COLL,
        query=vec,
        query_filter=Filter(
            must=[FieldCondition(key="study_id", match=MatchValue(value=study_id))]
        ),
        limit=top_k,
        with_payload=True,
        with_vectors=False,
    ).points
    return [{
        "score":     round(r.score, 3),
        "pseudonym": r.payload.get("pseudonym", ""),
        "tags":      r.payload.get("tags", {}),
        "text":      r.payload.get("text", ""),
        "question":  r.payload.get("question_ctx", ""),
    } for r in hits]
print("Embeddings + vector search loaded.")

# ================================================================
# PART E — PROMPTS
# ================================================================

INSIGHT_SYS = (
    "You are an expert UX researcher and qualitative analyst. "
    "Extract structured insights from interview transcripts. "
    "Rules: Be specific. Only extract what is genuinely present. Quotes must be verbatim. "
    "Return ONLY valid JSON. No markdown, no preamble."
)

THEME_SYS = (
    "You are a senior UX research strategist synthesising patterns across multiple interviews. "
    "Theme names MUST be specific insight statements that reveal the finding, NOT generic category labels. "
    "BAD theme names (too generic): 'Designers experience burnout', 'Research is undervalued', 'Designers need support'. "
    "GOOD theme names (specific insight): 'Protecting Monday morning focus time is the primary burnout defence', "
    "'Research findings are cited as validation rather than used to question direction', "
    "'Manager quality is the single largest variable in designer sustainability'. "
    "Theme names must be falsifiable observations, not category headings. "
    "A theme needs 2 or more participants unless it is a striking outlier worth surfacing. "
    "Return ONLY valid JSON. No markdown."
)

REC_SYS = (
    "You are a UX strategist turning research findings into actionable recommendations. "
    "Be specific: say what to do, not just what the problem is. "
    "Return ONLY valid JSON. No markdown."
)

SUMMARY_SYS = (
    "You are a UX research director writing an executive summary for stakeholders. "
    "Write clearly and without jargon. Return only the summary text, no JSON, no headers."
)

GAP_SYS = (
    "You are a rigorous UX research methodologist identifying gaps, "
    "unanswered questions, and contradictions. Return ONLY valid JSON."
)

def make_insight_prompt(study_purpose, pseudonym, tags, transcript_text):
    return (
        "Study Purpose: " + study_purpose + "\n"
        "Participant: " + pseudonym + " | Tags: " + tags + "\n\n"
        "Interview:\n" + transcript_text + "\n\n"
        "Extract all meaningful insights and return this JSON:\n"
        '{"insights": [{"insight_type": "pain_point|mental_model|workaround|delight|observation|surprise|need", '
        '"description": "analyst interpretation 1-2 sentences", '
        '"quote": "verbatim quote or null", '
        '"question_context": "triggering question or null", '
        '"is_surprising": false}]}'
    )

def make_theme_prompt(study_purpose, n_participants, observations):
    # Extract the actual pseudonym list from the observations so we can
    # explicitly tell the LLM which names to use — prevents hallucination
    import re
    found_names = list(dict.fromkeys(re.findall(r"^\[([^|\]]+)", observations, re.MULTILINE)))
    names_str = ", ".join(found_names) if found_names else "see observations"
    return (
        "Study Purpose: " + study_purpose + "\n"
        "Participants (" + str(n_participants) + " total): " + names_str + "\n\n"
        "Observations:\n" + observations + "\n\n"
        "Synthesise into themes and return this JSON:\n"
        '{"themes": [{"name": "specific insight-statement (not a category label)", '
        '"description": "2-3 sentence synthesis citing specific participant evidence", '
        '"participant_pseudonyms": ["EXACT names from observations who support this theme"], '
        '"confidence": "high|medium|low", '
        '"insight_types": ["pain_point"], '
        '"representative_quote": "verbatim quote from observations block or null", '
        '"representative_quote_participant": "exact pseudonym or null"}]}\n'
        "Confidence guide: high=6+ participants, medium=3-5, low=1-2.\n"
        "CRITICAL: participant_pseudonyms must use the EXACT names from the observations "
        "block above. Do not paraphrase or invent names."
    )

def make_rec_prompt(study_purpose, themes_block):
    return (
        "Study Purpose: " + study_purpose + "\n\n"
        "Themes:\n" + themes_block + "\n\n"
        "Generate recommendations and return this JSON:\n"
        '{"recommendations": [{"title": "short action-oriented title", '
        '"description": "what to do and why, 2-3 sentences", '
        '"rec_type": "design|strategy|research", '
        '"priority": "high|medium|low", '
        '"evidence_strength": "high|medium|low", '
        '"linked_theme_names": ["theme name"]}]}'
    )

def make_summary_prompt(study_purpose, n, themes_block, recs_block):
    return (
        "Study Purpose: " + study_purpose + "\n"
        "Participants: " + str(n) + "\n\n"
        "Top Themes:\n" + themes_block + "\n\n"
        "Top Recommendations:\n" + recs_block + "\n\n"
        "Write a 3-4 paragraph executive summary covering: "
        "1) what was studied and who was interviewed, "
        "2) the 2-3 most important findings, "
        "3) the most critical recommendation, "
        "4) any significant gaps or contradictions."
    )

def make_gap_prompt(study_purpose, script_questions, themes_block, insights_block):
    return (
        "Study Purpose: " + study_purpose + "\n"
        "Script Questions:\n" + script_questions + "\n\n"
        "Themes Found:\n" + themes_block + "\n\n"
        "Insights Sample:\n" + insights_block + "\n\n"
        "Return this JSON:\n"
        '{"gaps": [{"description": "what we do not know", '
        '"gap_type": "script_miss|thin_response|contradiction|unanswered", '
        '"related_question": "script question or null", '
        '"suggested_follow_up": "recommended question"}], '
        '"contradictions": [{"topic": "topic", '
        '"description": "what the contradiction is", '
        '"participant_a": "pseudonym", '
        '"participant_b": "pseudonym", '
        '"quote_a": "quote or null", '
        '"quote_b": "quote or null"}]}'
    )

print("Prompts loaded.")

# ================================================================
# PART F — PIPELINE
# ================================================================

def run_pipeline(study):
    print("\n" + "="*60)
    print("  UX Research Pipeline  |  " + study.study_id)
    print("="*60)

    # Parse raw transcripts
    for t in study.transcripts:
        if not t.utterances:
            t.utterances = parse_transcript(t.raw_text, t.participant.pseudonym)

    # Stage 1: Chunk + Embed
    print("\n[Stage 1] Chunking and embedding transcripts...")
    store_chunks(study.study_id, study.transcripts)

    # Stage 2: Per-interview insights
    print("\n[Stage 2] Extracting insights per interview...")
    all_insights = []
    for t in study.transcripts:
        print("  ->", t.participant.pseudonym, "...", end=" ", flush=True)
        tags_str = ", ".join(k + ": " + v for k,v in t.participant.tags.items()) or "none"
        try:
            result = llm_json(
                INSIGHT_SYS,
                make_insight_prompt(
                    study.study_purpose,
                    t.participant.pseudonym,
                    tags_str,
                    transcript_to_text(t)
                )
            )
            count = 0
            for r in result.get("insights", []):
                all_insights.append(Insight(
                    participant_id=t.participant.id,
                    pseudonym=t.participant.pseudonym,
                    participant_tags=t.participant.tags,
                    insight_type=r.get("insight_type", "observation"),
                    description=r.get("description", ""),
                    quote=r.get("quote"),
                    question_context=r.get("question_context"),
                    is_surprising=r.get("is_surprising", False),
                ))
                count += 1
            print(str(count) + " insights")
        except Exception as e:
            print("ERROR:", e)

    print("  Total insights:", len(all_insights))

    # Stage 3: Theme synthesis
    print("\n[Stage 3] Synthesising themes...")
    obs_lines = []
    for ins in all_insights:
        tags_str = ", ".join(k + "=" + v for k,v in ins.participant_tags.items())
        q = ' | Quote: "' + ins.quote + '"' if ins.quote else ""
        obs_lines.append("[" + ins.pseudonym + " | " + tags_str + " | " + ins.insight_type + "] " + ins.description + q)

    themes = []
    try:
        result = llm_json(
            THEME_SYS,
            make_theme_prompt(study.study_purpose, len(study.transcripts), "\n".join(obs_lines)),
            max_tokens=6000
        )
        for raw in result.get("themes", []):
            participants = raw.get("participant_pseudonyms", [])
            # Gather ONE best quote per participant first, then fill remaining slots.
            # This prevents the first participant alphabetically (Aisha) from
            # crowding out all others when all_insights is ordered by participant.
            _seen, _evidence_pool = set(), []
            for ins in all_insights:
                if ins.pseudonym in participants and ins.quote and ins.pseudonym not in _seen:
                    _evidence_pool.append({"pseudonym": ins.pseudonym, "quote": ins.quote, "tags": ins.participant_tags})
                    _seen.add(ins.pseudonym)
            # fill remaining slots with extra quotes from any participant
            for ins in all_insights:
                if len(_evidence_pool) >= 5:
                    break
                if ins.pseudonym in participants and ins.quote:
                    entry = {"pseudonym": ins.pseudonym, "quote": ins.quote, "tags": ins.participant_tags}
                    if entry not in _evidence_pool:
                        _evidence_pool.append(entry)
            evidence = _evidence_pool[:5]
            themes.append(Theme(
                name=raw.get("name", ""),
                description=raw.get("description", ""),
                participant_count=len(set(participants)),
                confidence=raw.get("confidence", "low"),
                insight_types=raw.get("insight_types", []),
                evidence=evidence,
                representative_quote=raw.get("representative_quote"),
                representative_quote_participant=raw.get("representative_quote_participant"),
            ))
        print("  " + str(len(themes)) + " themes identified")
    except Exception as e:
        print("  ERROR:", e)

    # Stage 4: Recommendations
    print("\n[Stage 4] Generating recommendations...")
    recs = []
    try:
        themes_block = "\n".join("- [" + t.confidence + " confidence] " + t.name + ": " + t.description for t in themes)
        result = llm_json(REC_SYS, make_rec_prompt(study.study_purpose, themes_block))
        for r in result.get("recommendations", []):
            recs.append(Recommendation(
                title=r.get("title", ""),
                description=r.get("description", ""),
                rec_type=r.get("rec_type", "design"),
                priority=r.get("priority", "medium"),
                evidence_strength=r.get("evidence_strength", "medium"),
                linked_theme_names=r.get("linked_theme_names", []),
            ))
        print("  " + str(len(recs)) + " recommendations generated")
    except Exception as e:
        print("  ERROR:", e)

    # Stage 5: Gaps + contradictions
    print("\n[Stage 5] Detecting gaps and contradictions...")
    gaps, contradictions = [], []
    try:
        script_qs     = "\n".join("- " + q for q in study.script_questions) or "No script provided."
        themes_block  = "\n".join("- " + t.name for t in themes)
        insights_block= "\n".join("[" + i.pseudonym + "] " + i.description for i in all_insights[:50])
        result = llm_json(GAP_SYS, make_gap_prompt(study.study_purpose, script_qs, themes_block, insights_block))
        for g in result.get("gaps", []):
            gaps.append(ResearchGap(
                description=g.get("description", ""),
                gap_type=g.get("gap_type", "unanswered"),
                related_question=g.get("related_question"),
                suggested_follow_up=g.get("suggested_follow_up"),
            ))
        for c in result.get("contradictions", []):
            contradictions.append(Contradiction(
                topic=c.get("topic", ""),
                description=c.get("description", ""),
                participant_a=c.get("participant_a", ""),
                participant_b=c.get("participant_b", ""),
                quote_a=c.get("quote_a"),
                quote_b=c.get("quote_b"),
            ))
        print("  " + str(len(gaps)) + " gaps,", len(contradictions), "contradictions")
    except Exception as e:
        print("  ERROR:", e)

    # Stage 6: Executive summary
    print("\n[Stage 6] Writing executive summary...")
    summary = ""
    try:
        themes_block = "\n".join("- " + t.name + " (" + t.confidence + ", " + str(t.participant_count) + " participants)" for t in themes)
        recs_block   = "\n".join("- [" + r.priority + "] " + r.title + ": " + r.description for r in recs[:5])
        summary = llm_complete(
            SUMMARY_SYS,
            make_summary_prompt(study.study_purpose, len(study.transcripts), themes_block, recs_block),
            temperature=0.3, max_tokens=800
        )
        print("  Done")
    except Exception as e:
        print("  ERROR:", e)
        summary = "Summary generation failed."

    print("\n" + "="*60)
    print("  Pipeline complete!")
    print("="*60)

    return StudyReport(
        study_id=study.study_id,
        study_purpose=study.study_purpose,
        participant_count=len(study.transcripts),
        executive_summary=summary,
        themes=themes,
        recommendations=recs,
        gaps=gaps,
        contradictions=contradictions,
        all_insights=all_insights,
    )

print("Pipeline loaded.")
print("\nAll systems ready! Move to Cell 4.")


Data models loaded.
Transcript parser loaded.
LLM client loaded.
Loading embedding model (first time ~30 sec)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embeddings + vector search loaded.
Prompts loaded.
Pipeline loaded.

All systems ready! Move to Cell 4.


---
## Cell 4 — Load Your Study

Run as-is to use the built-in sample (3 founder interviews).

To use your own data: replace the `raw_text` blocks with your transcripts.
Supported formats: `Moderator: text`, `[00:01] Speaker: text`, or plain paragraphs.

In [4]:
import os, re, io
from google.colab import files as colab_files
import ipywidgets as widgets
from IPython.display import display, clear_output

# ── Install docx parser if not already present ────────────────────────────
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'python-docx', '-q'])
import docx


# ================================================================
# PARSERS: .vtt  /  .docx  /  .txt
# ================================================================

def parse_vtt(content: str) -> str:
    """
    Convert WebVTT caption file into plain speaker-labeled transcript.
    Handles Zoom/Teams/Meet formats.
    VTT lines look like:
      00:01:23.456 --> 00:01:27.890
      Speaker Name: spoken text here
    """
    lines = content.splitlines()
    transcript_lines = []
    timestamp_re = re.compile(r'^\d{2}:\d{2}[:\.]\d{2}')
    prev_speaker = None
    buffer = []

    for line in lines:
        line = line.strip()
        # Skip header, blank lines, and timestamp lines
        if not line or line == 'WEBVTT' or timestamp_re.match(line) or line.isdigit():
            continue
        # Lines with speaker label: 'Name: text'
        speaker_match = re.match(r'^([A-Za-z][^:]{1,40}):\s+(.+)$', line)
        if speaker_match:
            spk, txt = speaker_match.groups()
            if spk == prev_speaker:
                buffer.append(txt)
            else:
                if prev_speaker and buffer:
                    transcript_lines.append(prev_speaker + ': ' + ' '.join(buffer))
                prev_speaker = spk
                buffer = [txt]
        else:
            # continuation line with no speaker label
            buffer.append(line)

    if prev_speaker and buffer:
        transcript_lines.append(prev_speaker + ': ' + ' '.join(buffer))

    return '\n'.join(transcript_lines)


def parse_docx(content: bytes) -> str:
    """Extract plain text from a .docx file (bytes)."""
    doc = docx.Document(io.BytesIO(content))
    return '\n'.join(p.text for p in doc.paragraphs if p.text.strip())


def parse_txt(content: bytes) -> str:
    """Decode a plain text file."""
    for enc in ('utf-8', 'utf-16', 'latin-1'):
        try:
            return content.decode(enc)
        except Exception:
            continue
    return content.decode('utf-8', errors='replace')


def file_to_raw_text(filename: str, content: bytes) -> str:
    ext = filename.lower().rsplit('.', 1)[-1]
    if ext == 'vtt':
        return parse_vtt(content.decode('utf-8', errors='replace'))
    elif ext == 'docx':
        return parse_docx(content)
    else:
        return parse_txt(content)


# ================================================================
# INPUT WIDGETS
# ================================================================

style   = {'description_width': '220px'}
layout  = widgets.Layout(width='750px')
ta_layout = widgets.Layout(width='750px', height='90px')

w_purpose = widgets.Textarea(
    description='Study Purpose *',
    placeholder='e.g. Understand how founders discover and adopt productivity tools',
    style=style, layout=ta_layout
)

w_rq = widgets.Textarea(
    description='Research Questions',
    placeholder='One per line e.g.\nWhat triggers a founder to look for a new tool?\nHow do they evaluate options?',
    style=style, layout=widgets.Layout(width='750px', height='110px')
)

w_sq = widgets.Textarea(
    description='Script Questions',
    placeholder='One per line — the exact questions asked in the interview',
    style=style, layout=widgets.Layout(width='750px', height='110px')
)

upload_btn = widgets.Button(
    description='Upload Transcript Files',
    button_style='primary',
    icon='upload',
    layout=widgets.Layout(width='220px', height='36px')
)

upload_out = widgets.Output()
uploaded_transcripts = []   # filled when files are uploaded

def on_upload_click(_):
    global uploaded_transcripts
    with upload_out:
        clear_output()
        print('Opening file picker — select one or more files...')
        raw_files = colab_files.upload()   # Colab native file picker
        uploaded_transcripts = []
        for fname, content in raw_files.items():
            pseudonym = fname.rsplit('.', 1)[0]   # filename without extension
            raw_text  = file_to_raw_text(fname, content)
            uploaded_transcripts.append(
                Transcript(
                    participant=Participant(pseudonym=pseudonym),
                    raw_text=raw_text
                )
            )
            print('Loaded:', fname, '->', len(raw_text), 'chars')
        print('\nTotal transcripts loaded:', len(uploaded_transcripts))

upload_btn.on_click(on_upload_click)

confirm_btn = widgets.Button(
    description='Confirm & Build Study',
    button_style='success',
    icon='check',
    layout=widgets.Layout(width='220px', height='36px')
)
confirm_out = widgets.Output()

def on_confirm(_):
    global study
    with confirm_out:
        clear_output()
        purpose = w_purpose.value.strip()
        if not purpose:
            print('ERROR: Study Purpose is required.')
            return
        if not uploaded_transcripts:
            print('ERROR: No transcripts uploaded yet. Click Upload Transcript Files first.')
            return
        rqs = [q.strip() for q in w_rq.value.strip().splitlines() if q.strip()]
        sqs = [q.strip() for q in w_sq.value.strip().splitlines() if q.strip()]
        study = Study(
            study_purpose      = purpose,
            research_questions = rqs,
            script_questions   = sqs,
            transcripts        = uploaded_transcripts,
        )
        print('Study built successfully!')
        print('  Purpose     :', study.study_purpose)
        print('  Participants:', len(study.transcripts),
              '->', ', '.join(t.participant.pseudonym for t in study.transcripts))
        print('  Res. questions :', len(study.research_questions))
        print('  Script questions:', len(study.script_questions))
        print()
        print('Ready! Run Cell 5 to start the analysis.')

confirm_btn.on_click(on_confirm)

# ── Render the form ───────────────────────────────────────────────────────
display(
    widgets.HTML('<h3 style="margin-bottom:4px">Study Details</h3>'),
    w_purpose,
    w_rq,
    w_sq,
    widgets.HTML('<h3 style="margin-top:12px;margin-bottom:4px">Transcript Files</h3>'),
    widgets.HTML('<p style="color:#555;font-size:13px">Supported: .vtt (Zoom/Teams/Meet captions), .docx (Word), .txt</p>'),
    upload_btn,
    upload_out,
    widgets.HTML('<h3 style="margin-top:12px;margin-bottom:4px">Confirm</h3>'),
    confirm_btn,
    confirm_out,
)


HTML(value='<h3 style="margin-bottom:4px">Study Details</h3>')

Textarea(value='', description='Study Purpose *', layout=Layout(height='90px', width='750px'), placeholder='e.…

Textarea(value='', description='Research Questions', layout=Layout(height='110px', width='750px'), placeholder…

Textarea(value='', description='Script Questions', layout=Layout(height='110px', width='750px'), placeholder='…

HTML(value='<h3 style="margin-top:12px;margin-bottom:4px">Transcript Files</h3>')

...

Button(button_style='primary', description='Upload Transcript Files', icon='upload', layout=Layout(height='36p…

Output()

HTML(value='<h3 style="margin-top:12px;margin-bottom:4px">Confirm</h3>')

Button(button_style='success', description='Confirm & Build Study', icon='check', layout=Layout(height='36px',…

Output()

---
## Cell 5 — Run the Full Pipeline

Takes about 60-90 seconds.

In [7]:
report = run_pipeline(study)


  UX Research Pipeline  |  e35e0ae6-15f

[Stage 1] Chunking and embedding transcripts...
  Stored 69 chunks in vector DB

[Stage 2] Extracting insights per interview...
  -> Aisha ... 12 insights
  -> Ben ... 10 insights
  -> Cleo ... 11 insights
  -> Dev ... 8 insights
  -> Erin ... 9 insights
  -> Felix ... 10 insights
  -> Grace ... 10 insights
  Total insights: 70

[Stage 3] Synthesising themes...
  6 themes identified

[Stage 4] Generating recommendations...
  6 recommendations generated

[Stage 5] Detecting gaps and contradictions...
  4 gaps, 2 contradictions

[Stage 6] Writing executive summary...
  Done

  Pipeline complete!


---
## Cell 6 — View Your Results

In [8]:
def section(title, emoji=""):
    print("\n" + "-"*60)
    print("  " + emoji + "  " + title)
    print("-"*60)

CONF_ICON = {"high": "GREEN", "medium": "YELLOW", "low": "RED"}

section("EXECUTIVE SUMMARY", "📋")
print(report.executive_summary)

section("THEMES (" + str(len(report.themes)) + " found)", "🎯")
for i, t in enumerate(report.themes, 1):
    print("\n  Theme " + str(i) + ": " + t.name)
    print("  Confidence: " + t.confidence + "  |  Participants: " + str(t.participant_count))
    print("  " + t.description)
    if t.representative_quote:
        print("\n  Quote: \"" + t.representative_quote + "\"")
        print("  -- " + (t.representative_quote_participant or "Participant"))
    for ev in t.evidence[:2]:
        if ev.get("quote"):
            tags_str = ", ".join(k + ": " + v for k,v in ev.get("tags",{}).items())
            print("  * \"" + ev["quote"] + "\" [" + ev["pseudonym"] + " | " + tags_str + "]")

section("RECOMMENDATIONS (" + str(len(report.recommendations)) + " generated)", "💡")
for i, r in enumerate(report.recommendations, 1):
    print("\n  [" + r.priority.upper() + "] " + r.title + " (" + r.rec_type + ")")
    print("  " + r.description)
    if r.linked_theme_names:
        print("  Evidence from: " + ", ".join(r.linked_theme_names))

if report.contradictions:
    section("CONTRADICTIONS (" + str(len(report.contradictions)) + " found)", "⚡")
    for c in report.contradictions:
        print("\n  Topic: " + c.topic)
        print("  " + c.description)
        if c.quote_a: print("  " + c.participant_a + ": \"" + c.quote_a + "\"")
        if c.quote_b: print("  " + c.participant_b + ": \"" + c.quote_b + "\"")

if report.gaps:
    section("RESEARCH GAPS (" + str(len(report.gaps)) + " identified)", "🔍")
    for g in report.gaps:
        print("\n  [" + g.gap_type + "] " + g.description)
        if g.suggested_follow_up:
            print("  Follow-up: " + g.suggested_follow_up)

section("INSIGHT BREAKDOWN", "🧩")
by_type = {}
for ins in report.all_insights:
    by_type[ins.insight_type] = by_type.get(ins.insight_type, 0) + 1
for itype, count in sorted(by_type.items(), key=lambda x: -x[1]):
    print("  " + itype.ljust(20) + " |" + "#"*count + "| (" + str(count) + ")")

print("\n  Report ID: " + report.study_id + "  |  " + report.generated_at[:19])


------------------------------------------------------------
  📋  EXECUTIVE SUMMARY
------------------------------------------------------------
Our study aimed to understand the experiences of mid-career UX designers at B2B SaaS companies, focusing on burnout, workload management, and the gap between research findings and product decisions. We conducted in-depth interviews with 7 participants, gathering valuable insights into their daily challenges and frustrations. The study revealed several key themes, but some findings stood out as particularly significant.

The most important findings from our study highlight the crucial role of dedicated time for focused work, the frequent disregard of research findings in product decisions, and the impact of managerial support on preventing burnout. Notably, 6 out of 7 participants reported that their research findings are often ignored or misinterpreted in product decisions, leading to frustration and demotivation. Additionally, the importance

---
## Cell 7 — Ask Questions About Your Transcripts

Change the `query` to anything. Searches by *meaning*, not keywords.

Examples: `"pricing frustrations"`, `"how founders discover tools"`, `"onboarding problems"`

In [ ]:
query = "pricing frustrations and tool costs"  # change this

print("Query:", query)
print("-"*60)
results = semantic_search(query, study.study_id, top_k=5)
for i, r in enumerate(results, 1):
    tags_str = ", ".join(k + ": " + v for k,v in r["tags"].items())
    print("\nResult " + str(i) + " | " + r["pseudonym"] + " [" + tags_str + "] | Score: " + str(r["score"]))
    if r.get("question"):
        print("  Q: " + r["question"])
    print("  " + r["text"])

Query: pricing frustrations and tool costs
------------------------------------------------------------


AttributeError: 'QdrantClient' object has no attribute 'search'

---
## Cell 8 — Submit Feedback

Rate any output. This builds a preference dataset for improving your prompts.

In [9]:
feedback_log = []

def submit_feedback(target_type, target_id, original_text, rating, edited_text=None, comment=None):
    entry = {
        "id":            uid(),
        "study_id":      report.study_id,
        "target_type":   target_type,
        "target_id":     target_id,
        "rating":        rating,
        "original_text": original_text,
        "edited_text":   edited_text,
        "comment":       comment,
        "created_at":    datetime.now(tz=timezone.utc).isoformat(),
    }
    feedback_log.append(entry)
    print("Feedback saved: [" + rating + "] on " + target_type)
    return entry

# Rate a theme
if report.themes:
    submit_feedback(
        target_type="theme",
        target_id=report.themes[0].id,
        original_text=report.themes[0].name,
        rating="positive",
        comment="Accurate and well-named"
    )

# Rate a recommendation (edit the description if it needs improvement)
if report.recommendations:
    submit_feedback(
        target_type="recommendation",
        target_id=report.recommendations[0].id,
        original_text=report.recommendations[0].description,
        rating="edited",
        edited_text="[Your improved version here]",
        comment="Too vague, needs a more specific action"
    )

print("Total feedback entries:", len(feedback_log))

Feedback saved: [positive] on theme
Feedback saved: [edited] on recommendation
Total feedback entries: 2


---
## Cell 9 — Export and Download Report

In [ ]:
import json
from dataclasses import asdict
from google.colab import files

def to_dict(obj):
    if hasattr(obj, "__dataclass_fields__"):
        return {k: to_dict(v) for k, v in asdict(obj).items()}
    elif isinstance(obj, list):
        return [to_dict(i) for i in obj]
    elif isinstance(obj, dict):
        return {k: to_dict(v) for k, v in obj.items()}
    return obj

output = {"report": to_dict(report), "feedback": feedback_log}
filename = "ux_report_" + report.study_id + ".json"
with open(filename, "w") as f:
    json.dump(output, f, indent=2)

print("Report saved as", filename)
files.download(filename)

Report saved as ux_report_b9e4aa1b-854.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## Cell 10 — Add Your Own Transcripts

Replace the content below with your own participant data, then re-run Cell 5.

In [ ]:
new_transcript = Transcript(
    participant=Participant(
        pseudonym="Jordan",
        tags={"role": "founder", "company_size": "12 people", "stage": "seed"}
    ),
    raw_text=(
        "Moderator: Walk me through the last time you needed a new tool.\n"
        "Jordan: We needed async video for our remote team across 3 timezones.\n"
        "Moderator: How did you find your options?\n"
        "Jordan: Someone in a founders Slack group mentioned two alternatives. That is where I get"
        " most tool recommendations now, curated communities of founders at similar stages.\n"
        "Moderator: What made you pick the one you did?\n"
        "Jordan: Best free tier and onboarding was genuinely good. Up and running in 5 minutes.\n"
        "Moderator: Have you stopped using any tools recently?\n"
        "Jordan: We killed our CRM because it was too complex for our stage. Went back to a spreadsheet.\n"
        "Moderator: What is your biggest frustration?\n"
        "Jordan: Integration hell. Every tool wants to be the hub and none talk to each other without Zapier."
    )
)

study.transcripts.append(new_transcript)
print("Added", new_transcript.participant.pseudonym)
print("Study now has", len(study.transcripts), "participants")
print("Re-run Cell 5 to re-analyse with the new transcript included.")